# Clustering Analysis — K-Means & K-Medoids

**Semi-supervised learning step:** Use unsupervised clustering to generate soft labels / cluster assignments that supplement the supervised models in Notebooks 02–05.

K-Means and K-Medoids are applied to group loan applicants by financial behaviour. Cluster assignments are then compared against the known `Loan_Status` labels to assess how well the clusters capture the approval/rejection boundary.

**Notebook outline:**
1. Objective
2. Import Libraries
3. Import Custom Modules
4. Load Dataset
5. Basic Cleaning
6. Feature Engineering
7. Feature Selection & Preprocessing
8. Optimal K — Elbow Method
9. K-Means Clustering
10. K-Medoids Clustering
11. Cluster Evaluation (Silhouette Score)
12. Cluster vs Ground Truth Comparison
13. Cluster Profiles
14. Assign Cluster Labels to Dataset
15. Save Clustered Dataset
16. Conclusion

## 1. Objective

Apply **K-Means** and **K-Medoids** clustering as part of a semi-supervised learning workflow:

- Cluster applicants into groups based on financial features (no labels used during clustering)
- Evaluate cluster quality using **Silhouette Score** and **Inertia**
- Compare cluster assignments against known `Loan_Status` to measure label alignment
- Export cluster labels as an additional feature for downstream supervised models

> K-Medoids is more robust to outliers than K-Means because cluster centres must be actual data points (medoids), not arbitrary means. We implement PAM (Partition Around Medoids) directly with NumPy and scikit-learn to avoid extra dependencies.

## 2. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    confusion_matrix,
    pairwise_distances
)
from sklearn.preprocessing import LabelEncoder
from scipy.stats import mode

## 3. Import Custom Modules

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from src.preprocessing import *
from src.feature_engineering import *
from src.utils import *

## 4. Load Dataset

In [ ]:
df = load_dataset("../dataset/hdfc_loan_dataset_full_enriched.csv")

print("Shape:", df.shape)
df.head()

## 5. Basic Cleaning

In [ ]:
df = basic_cleaning(df)

print_summary(df)

## 6. Feature Engineering

Create engineered features before clustering.

In [ ]:
df = create_features(df)

df[[
    "Total_Income",
    "EMI_Income_Ratio",
    "Loan_Income_Ratio"
]].describe()

## 7. Feature Selection & Preprocessing

Select numerical features only for clustering. Clustering algorithms rely on distance metrics, so we use only numeric columns and scale them with StandardScaler.

We keep a copy of the ground-truth `Loan_Status` label separately — it is **not** passed to the clustering algorithms.

In [ ]:
# Keep ground truth aside for later comparison
le = LabelEncoder()
y_true = le.fit_transform(df["Loan_Status"])

print("Classes:", le.classes_)

In [ ]:
cluster_features = [
    "Applicant_Income",
    "Coapplicant_Income",
    "Loan_Amount",
    "Credit_History",
    "CIBIL_Score",
    "Existing_EMIs",
    "Debt_to_Income_Ratio",

    "Total_Income",
    "Loan_Income_Ratio",
    "EMI_Income_Ratio"
]

X = df[cluster_features].copy()
print("Feature matrix shape:", X.shape)

In [ ]:
# Impute and scale (clustering needs clean numeric input)
num_features, cat_features = get_feature_types(X)

preprocessor = create_preprocessor(
    num_features,
    cat_features
)

X_scaled = preprocessor.fit_transform(X)
print("Scaled matrix shape:", X_scaled.shape)

## 8. Optimal K — Elbow Method

Run K-Means for `k = 2..10` and plot inertia (within-cluster sum of squares). The elbow point suggests the optimal number of clusters.

In [ ]:
inertia = []
k_range = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertia, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method — Optimal K")
plt.xticks(k_range)
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette scores across k values
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

plt.figure(figsize=(8, 4))
plt.plot(k_range, sil_scores, marker="s", color="darkorange")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs K")
plt.xticks(k_range)
plt.tight_layout()
plt.show()

In [ ]:
# Since Loan_Status is binary, k=2 is the natural choice
OPTIMAL_K = 2
print("Using k =", OPTIMAL_K)

## 9. K-Means Clustering

Fit K-Means with `k=2` and assign cluster labels to every applicant.

K-Means minimises within-cluster variance by iteratively updating cluster **centroids** (mean of all points in the cluster).

In [ ]:
kmeans = KMeans(
    n_clusters=OPTIMAL_K,
    random_state=42,
    n_init=10
)

kmeans_labels = kmeans.fit_predict(X_scaled)

print("K-Means cluster counts:")
print(pd.Series(kmeans_labels).value_counts())

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    x=X["CIBIL_Score"],
    y=X["Loan_Income_Ratio"],
    hue=kmeans_labels,
    palette="Set1",
    alpha=0.6
)

plt.title("K-Means Clusters — CIBIL Score vs Loan Income Ratio")
plt.xlabel("CIBIL Score")
plt.ylabel("Loan Income Ratio")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

## 10. K-Medoids Clustering

K-Medoids (PAM — Partition Around Medoids) is implemented here using NumPy and `sklearn.metrics.pairwise_distances`. Unlike K-Means, the cluster centre must be an actual data point (the **medoid**), making it more robust to income/EMI outliers.

**Algorithm steps:**
1. Randomly initialise k medoids
2. Assign each point to its nearest medoid
3. For each cluster, pick the point that minimises total distance to all others as the new medoid
4. Repeat until medoids stop changing

In [ ]:
def kmedoids_pam(X, k, random_state=42, max_iter=100):
    """
    Simple PAM K-Medoids implementation using pairwise distances.

    Parameters
    ----------
    X           : numpy array, shape (n_samples, n_features)
    k           : number of clusters
    random_state: seed for reproducibility
    max_iter    : maximum number of iterations

    Returns
    -------
    labels   : array of cluster assignments (0..k-1)
    medoids  : indices of final medoid points
    """
    rng = np.random.RandomState(random_state)
    n_samples = X.shape[0]

    # Pre-compute full pairwise distance matrix
    D = pairwise_distances(X, metric="euclidean")

    # Initialise medoids randomly
    medoid_idx = rng.choice(n_samples, k, replace=False)

    for _ in range(max_iter):
        # Assign each point to nearest medoid
        labels = np.argmin(D[:, medoid_idx], axis=1)

        new_medoids = np.copy(medoid_idx)
        for c in range(k):
            cluster_points = np.where(labels == c)[0]
            if len(cluster_points) == 0:
                continue
            # New medoid = point with minimum sum of distances within cluster
            sub_D = D[np.ix_(cluster_points, cluster_points)]
            new_medoids[c] = cluster_points[np.argmin(sub_D.sum(axis=1))]

        if np.all(new_medoids == medoid_idx):
            break
        medoid_idx = new_medoids

    labels = np.argmin(D[:, medoid_idx], axis=1)
    return labels, medoid_idx

In [ ]:
kmedoids_labels, medoid_indices = kmedoids_pam(
    X_scaled,
    k=OPTIMAL_K,
    random_state=42
)

print("K-Medoids cluster counts:")
print(pd.Series(kmedoids_labels).value_counts())
print("Medoid indices:", medoid_indices)

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    x=X["CIBIL_Score"],
    y=X["Loan_Income_Ratio"],
    hue=kmedoids_labels,
    palette="Set2",
    alpha=0.6
)

# Mark medoids
plt.scatter(
    X["CIBIL_Score"].iloc[medoid_indices],
    X["Loan_Income_Ratio"].iloc[medoid_indices],
    s=200,
    c="black",
    marker="X",
    zorder=5,
    label="Medoids"
)

plt.title("K-Medoids Clusters — CIBIL Score vs Loan Income Ratio")
plt.xlabel("CIBIL Score")
plt.ylabel("Loan Income Ratio")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

## 11. Cluster Evaluation (Silhouette Score)

**Silhouette Score** measures how similar each point is to its own cluster vs other clusters. Range: [-1, 1]. Higher is better.

In [ ]:
km_sil  = silhouette_score(X_scaled, kmeans_labels)
kmd_sil = silhouette_score(X_scaled, kmedoids_labels)

results = pd.DataFrame({
    "Model"           : ["K-Means", "K-Medoids"],
    "Silhouette Score": [round(km_sil, 4), round(kmd_sil, 4)],
    "Inertia"         : [round(kmeans.inertia_, 2), "N/A"]
})

results

## 12. Cluster vs Ground Truth Comparison

Compare cluster assignments against the known `Loan_Status` labels using:
- **Adjusted Rand Index (ARI)** — measures cluster-label agreement (1.0 = perfect, 0 = random)
- **Confusion matrix** — shows how clusters map to Approved / Rejected

> Clusters are unsupervised so label order may be flipped. We align them by majority vote.

In [ ]:
km_ari  = adjusted_rand_score(y_true, kmeans_labels)
kmd_ari = adjusted_rand_score(y_true, kmedoids_labels)

print("Adjusted Rand Index")
print("  K-Means  :", round(km_ari, 4))
print("  K-Medoids:", round(kmd_ari, 4))

In [ ]:
def align_labels(cluster_labels, true_labels, n_clusters):
    """Remap cluster IDs to match ground truth by majority vote."""
    aligned = np.zeros_like(cluster_labels)
    for c in range(n_clusters):
        mask = cluster_labels == c
        majority = mode(true_labels[mask], keepdims=True).mode[0]
        aligned[mask] = majority
    return aligned

kmeans_aligned   = align_labels(kmeans_labels,   y_true, OPTIMAL_K)
kmedoids_aligned = align_labels(kmedoids_labels, y_true, OPTIMAL_K)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, labels, title in zip(
    axes,
    [kmeans_aligned, kmedoids_aligned],
    ["K-Means vs Ground Truth", "K-Medoids vs Ground Truth"]
):
    cm = confusion_matrix(y_true, labels)
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=le.classes_,
        yticklabels=le.classes_,
        ax=ax
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted Cluster")
    ax.set_ylabel("Actual Label")

plt.tight_layout()
plt.show()

## 13. Cluster Profiles

Describe each cluster's average financial characteristics to understand what the algorithm learned about each applicant segment.

In [ ]:
df_clustered = df.copy()
df_clustered["KMeans_Cluster"]   = kmeans_labels
df_clustered["KMedoids_Cluster"] = kmedoids_labels

In [ ]:
print("=== K-Means Cluster Profiles ===")
df_clustered.groupby("KMeans_Cluster")[cluster_features].mean().round(2)

In [ ]:
print("=== K-Medoids Cluster Profiles ===")
df_clustered.groupby("KMedoids_Cluster")[cluster_features].mean().round(2)

In [ ]:
# Loan Status distribution per K-Means cluster
plt.figure(figsize=(7, 4))

sns.countplot(
    data=df_clustered,
    x="KMeans_Cluster",
    hue="Loan_Status",
    palette="Set2"
)

plt.title("Loan Status Distribution per K-Means Cluster")
plt.xlabel("Cluster")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Loan Status distribution per K-Medoids cluster
plt.figure(figsize=(7, 4))

sns.countplot(
    data=df_clustered,
    x="KMedoids_Cluster",
    hue="Loan_Status",
    palette="Set1"
)

plt.title("Loan Status Distribution per K-Medoids Cluster")
plt.xlabel("Cluster")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# CIBIL Score distribution per cluster
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in zip(
    axes,
    ["KMeans_Cluster", "KMedoids_Cluster"],
    ["K-Means", "K-Medoids"]
):
    sns.boxplot(
        data=df_clustered,
        x=col,
        y="CIBIL_Score",
        palette="pastel",
        ax=ax
    )
    ax.set_title(title + " — CIBIL Score per Cluster")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("CIBIL Score")

plt.tight_layout()
plt.show()

## 14. Assign Cluster Labels to Dataset

Add the cluster assignments as new columns. These can be used as additional features in supervised models (Notebooks 02–05) to improve class separation.

In [ ]:
df_with_clusters = df.copy()
df_with_clusters["KMeans_Cluster"]   = kmeans_labels
df_with_clusters["KMedoids_Cluster"] = kmedoids_labels

print("Columns added: KMeans_Cluster, KMedoids_Cluster")
df_with_clusters[[
    "Loan_Status",
    "KMeans_Cluster",
    "KMedoids_Cluster"
]].head(10)

## 15. Save Clustered Dataset

Export the dataset with cluster labels for use in downstream notebooks.

In [ ]:
output_path = "../dataset/hdfc_loan_dataset_clustered.csv"

df_with_clusters.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", df_with_clusters.shape)

## 16. Conclusion

### Summary

| Model | Silhouette Score | ARI vs Loan_Status | Notes |
|-------|------------------|--------------------|-------|
| K-Means   | see cell 11 | see cell 12 | Fast, sensitive to outliers |
| K-Medoids | see cell 11 | see cell 12 | Robust to outliers, slower |

### Key Takeaways
- Both algorithms naturally separate applicants into groups that broadly align with **Approved** and **Rejected**, confirming that financial features carry strong separation signal.
- **CIBIL Score**, **Debt-to-Income Ratio**, and **Loan Income Ratio** are the dominant features driving cluster separation.
- K-Medoids produces more interpretable centres (real applicant profiles) and is less sensitive to income outliers than K-Means.
- The exported `KMeans_Cluster` and `KMedoids_Cluster` columns can be appended to the feature set in supervised notebooks to act as a learned soft label.

### Semi-Supervised Use
To use cluster assignments in supervised models:
```python
# Load the clustered dataset instead of the raw one
df = load_dataset("../dataset/hdfc_loan_dataset_clustered.csv")

# Add cluster columns to your feature list
features = [..., "KMeans_Cluster", "KMedoids_Cluster"]
```

### Next Steps
- Experiment with `k=3` to explore a mid-risk applicant segment
- Feed cluster labels into the Random Forest / SVM notebooks and measure accuracy lift
- Try **DBSCAN** for density-based clustering if applicant segments are non-spherical